In [ ]:
# Wasserstein risk index (distribution shift) for futures

This notebook demonstrates a small end-to-end research pipeline:

- Load *continuous* daily futures prices (placeholder fake generator for now)
- Compute daily log returns
- Build the Wasserstein distribution-shift index \(W_t\) from cross-sectional returns
- Build a core panel with realized vol + rolling largest eigenvalue
- Run a simple HAC regression, an event study, and a conditioning experiment

**TODO**: Replace the fake loader in `algogators_wrisk.data.load_continuous_futures_prices` with your internal `algogators-data` call.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make repo root importable when running from notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from algogators_wrisk import config
from algogators_wrisk import data, features, analysis

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

np.random.seed(0)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

# Research parameters (override defaults here if desired)
universe = config.UNIVERSE
start_date = config.START_DATE
end_date = config.END_DATE

rv_past_window = config.RV_PAST_WINDOW
rv_future_window = config.RV_FUTURE_WINDOW
lambda1_window = config.LAMBDA1_WINDOW

w_event_q = config.W_EVENT_QUANTILE
exposure_on_event = config.EXPOSURE_ON_EVENT
hac_lags = config.HAC_LAGS

print(f"Universe size: {len(universe)}")
print(f"Date range: {start_date} → {end_date}")
print(f"RV windows (past/future): {rv_past_window}/{rv_future_window}")
print(f"Lambda1 window: {lambda1_window}")
print(f"Event quantile: {w_event_q}")

In [ ]:
# 1) Load continuous futures prices (FAKE placeholder)
prices = data.load_continuous_futures_prices(
    universe=universe,
    start_date=start_date,
    end_date=end_date,
    seed=42,
)

display(prices.head())
print(prices.shape)

In [ ]:
# 2) Compute daily log returns
returns = data.compute_log_returns(prices)
ret_matrix = features.build_return_matrix(returns, universe=universe)

display(ret_matrix.head())
print(ret_matrix.shape)

In [ ]:
# 3) Compute W_t, realized volatility, and rolling lambda1 via a single panel
panel = analysis.build_core_panel(
    ret_matrix,
    rv_past_window=rv_past_window,
    rv_future_window=rv_future_window,
    lambda1_window=lambda1_window,
    annualize_rv=True,
)

display(panel.tail())
print(panel.isna().mean().sort_values(ascending=False).head(10))

# 4) Plot W_t, realized vol, and lambda1
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

panel["W"].plot(ax=axes[0], color="black", lw=1)
axes[0].set_title("Wasserstein distribution-shift index (W)")

panel[["rv_past", "rv_future"]].plot(ax=axes[1], lw=1)
axes[1].set_title("Realized volatility (past vs future)")

panel["lambda1"].plot(ax=axes[2], color="tab:purple", lw=1)
axes[2].set_title("Rolling largest correlation eigenvalue (lambda1)")

plt.tight_layout()
plt.show()

In [ ]:
# 5) Regression: rv_future ~ W + rv_past + lambda1 (HAC SEs)
res = analysis.run_rv_regression(panel, hac_lags=hac_lags)
print(res.summary())

# 6) Event study: average market return path around high-W days
es = analysis.make_event_study_dataset(
    panel,
    value_col="mkt_ret",
    quantile=w_event_q,
    pre=10,
    post=10,
    min_gap=5,
)

print(f"Identified {len(es.events)} event days (after de-clustering).")
display(es.stacked.head())

plt.figure(figsize=(10, 4))
es.avg_path.plot(marker="o")
plt.axvline(0, color="black", lw=1)
plt.title("Event study: average mkt_ret around high-W events")
plt.xlabel("tau (days from event)")
plt.ylabel("avg return")
plt.tight_layout()
plt.show()

In [ ]:
# 7) Strategy conditioning: reduce exposure on high-W days
strat = analysis.run_strategy_conditioning_experiment(
    panel,
    quantile=w_event_q,
    exposure_on_event=exposure_on_event,
)

display(strat.head())

plt.figure(figsize=(12, 4))
strat[["baseline_cum", "conditioned_cum"]].plot(lw=1.5)
plt.title("Cumulative PnL (baseline vs conditioned)")
plt.xlabel("date")
plt.ylabel("cum pnl (sum of daily returns)")
plt.tight_layout()
plt.show()

summary = pd.Series(
    {
        "baseline_mean": strat["baseline_pnl"].mean(),
        "baseline_vol": strat["baseline_pnl"].std(),
        "conditioned_mean": strat["conditioned_pnl"].mean(),
        "conditioned_vol": strat["conditioned_pnl"].std(),
        "event_rate": strat["is_event"].mean(),
    }
)
print(summary)

## Notes

- The price loader is currently **fake** and deterministic.
- Once you wire real prices, everything downstream should work unchanged.

In [ ]:
# Normalize the features
X_normalized = features.normalize_features(df_returns)
X_normalized_df = pd.DataFrame(
    X_normalized,
    columns=[f'{col}_norm' for col in df_returns.columns]
)

print("Normalized Features Statistics:")
print(X_normalized_df.describe())

# Scale features to [0, 1] range
X_scaled = features.scale_features(df_returns)
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=[f'{col}_scaled' for col in df_returns.columns]
)

print("\nScaled Features Statistics:")
print(X_scaled_df.describe())

In [ ]:
# Visualize data distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(df_returns.columns):
    axes[i].hist(df_returns[col], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution: {col}')
    axes[i].set_xlabel('Returns')
    axes[i].set_ylabel('Frequency')

fig.suptitle('Asset Return Distributions', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
print("Configuration Settings:")
print(f"Data Path: {config.DATA_PATH}")
print(f"Output Path: {config.OUTPUT_PATH}")
print(f"Random Seed: {config.RANDOM_SEED}")
print(f"Test Size: {config.TEST_SIZE}")
print(f"Default Wasserstein P: {config.DEFAULT_WASSERSTEIN_P}")

# Set random seed for reproducibility
np.random.seed(config.RANDOM_SEED)

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import from algogators_wrisk package
from algogators_wrisk import config
from algogators_wrisk import data
from algogators_wrisk import features
from algogators_wrisk import analysis

# Set visualization style
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)